In [3]:
!pip install -q tpot

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from tpot import TPOTClassifier

df = pd.read_csv('kidney_disease 2.csv')

df.replace('?', np.nan, inplace=True)

if 'id' in df.columns:
    df = df.drop('id', axis=1)


cols_to_convert = ['pcv', 'wc', 'rc']
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'classification'
X = df.drop(target_col, axis=1)
y = df[target_col]

le_target = LabelEncoder()
y = le_target.fit_transform(y)


for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(rf, param_grid, cv=3)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred_manual = best_model.predict(X_test)

print("\n🔹 Manual Model Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_manual))
print(classification_report(y_test, y_pred_manual))

tpot = TPOTClassifier(
    generations=2,          # small to avoid crash
    population_size=5,
    random_state=42
)

tpot.fit(X_train, y_train)

y_pred_auto = tpot.predict(X_test)

print("\n🤖 AutoML Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_auto))
print(classification_report(y_test, y_pred_auto))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(



🔹 Manual Model Results:
Accuracy: 0.975
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        50
           2       1.00      0.93      0.97        30

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80



/usr/local/lib/python3.12/dist-packages/tpot/tpot_estimator/estimator.py:458: UserWarning: Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.
  warnings.warn("Both generations and max_time_mins are set. TPOT will terminate when the first condition is met.")
INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:33723
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:33457'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:33159 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:33159
INFO:distributed.core:Starting established connection to tcp://127.0.0.1


🤖 AutoML Results:
Accuracy: 0.95
              precision    recall  f1-score   support

           0       0.96      0.96      0.96        50
           2       0.93      0.93      0.93        30

    accuracy                           0.95        80
   macro avg       0.95      0.95      0.95        80
weighted avg       0.95      0.95      0.95        80

